In [1]:
%load_ext autoreload

from pathlib import Path
import sys; sys.path.append('../')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Display all columns in the DataFrame
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

sns.set_theme(style='whitegrid')



In [2]:
# Load the most recently touched simulation output folder under ../data/
repo_root = Path.cwd().resolve().parent
data_root = repo_root / 'data'

sim_dirs = [
    p for p in data_root.iterdir()
    if p.is_dir() and (p / 'data').is_dir()
]

if not sim_dirs:
    raise FileNotFoundError("No simulation output folders with a nested data/ directory found under ../data")

latest_sim_dir = max(sim_dirs, key=lambda p: p.stat().st_mtime)
data_dir = latest_sim_dir / 'data'

print(f"Using simulation folder: {latest_sim_dir}")
print("Available files:", sorted(p.name for p in data_dir.glob('*')))

# Current exporter format (src/sim/data_exporter.py):
# - products.parquet, stores.parquet, timeseries.parquet, run_log.json
# Keep compatibility variables used by this notebook.
products_df = pd.read_parquet(data_dir / 'products.parquet')

stores_path = data_dir / 'stores.parquet'
timeseries_path = data_dir / 'timeseries.parquet'

if stores_path.exists() and timeseries_path.exists():
    agents_df = pd.read_parquet(stores_path).rename(columns={'store_id': 'agent_id'})
    dynamic_product_data_df = pd.read_parquet(timeseries_path).rename(columns={'store_id': 'agent_id'})
    dynamic_agent_data_df = (
        dynamic_product_data_df[['agent_id', 'simulation_step']]
        .drop_duplicates()
        .sort_values(['agent_id', 'simulation_step'])
        .reset_index(drop=True)
    )
    related_products_df = pd.DataFrame()
else:
    # Legacy format fallback
    agents_df = pd.read_parquet(data_dir / 'agents.parquet')
    dynamic_agent_data_df = pd.read_parquet(data_dir / 'dynamic_agent_data.parquet')
    dynamic_product_data_df = pd.read_parquet(data_dir / 'dynamic_product_data.parquet')
    related_products_df = pd.read_parquet(data_dir / 'related_products.parquet')

Using simulation folder: /Users/mislavjordanic/Documents/projects/personal_projects/supply_chain_simulator/data/llm_world_100
Available files: ['products.parquet', 'run_log.json', 'stores.parquet', 'timeseries.parquet']


In [3]:
agents_df.columns

Index(['agent_id', 'template_id', 'region', 'init_seed', 'policy_type'], dtype='str')

In [4]:
products_df.columns

Index(['product_id', 'name', 'category', 'base_price', 'unit_cost', 'seasonality'], dtype='str')

In [5]:
dynamic_product_data_df[dynamic_product_data_df['total_cost'] != dynamic_product_data_df['holding_cost']]

,simulation_step,simulation_date,agent_id,product_id,inventory,demand,sales,order_quantity,outstanding_orders,promotion_status,active_status,price,revenue,total_cost,holding_cost,profit
21,21,2024-01-22,0,P0000,0,7,0,57,57,Regular Price,True,55.173142,0.00000,1453.000,0.000,-1453.00000
30,30,2024-01-31,0,P0000,0,10,0,57,57,Regular Price,True,52.999100,0.00000,1453.000,0.000,-1453.00000
38,38,2024-02-08,0,P0000,0,5,0,57,57,Regular Price,True,50.910724,0.00000,1453.000,0.000,-1453.00000
46,46,2024-02-16,0,P0000,0,19,0,57,57,Regular Price,True,40.012886,0.00000,1453.000,0.000,-1453.00000
68,17,2024-01-18,0,P0001,0,7,1,56,56,Regular Price,True,29.991060,29.99106,645.000,0.000,-615.00894
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15106,10,2024-01-11,2,P0088,26,4,4,27,27,Regular Price,True,91.476000,365.90400,822.424,8.424,-456.52000
15152,5,2024-01-06,2,P0089,10,18,18,29,29,Regular Price,True,29.700000,534.60000,317.960,0.960,216.64000
15567,12,2024-01-13,2,P0097,29,5,5,23,23,Regular Price,True,36.655740,183.27870,318.480,3.480,-135.20130
15616,10,2024-01-11,2,P0098,28,4,4,25,25,Regular Price,True,40.968180,163.87272,363.696,3.696,-199.82328


In [6]:
dynamic_agent_data_df.columns

Index(['agent_id', 'simulation_step'], dtype='str')

In [7]:
dynamic_product_data_df.columns

Index(['simulation_step', 'simulation_date', 'agent_id', 'product_id', 'inventory', 'demand', 'sales', 'order_quantity', 'outstanding_orders',
       'promotion_status', 'active_status', 'price', 'revenue', 'total_cost', 'holding_cost', 'profit'],
      dtype='str')

In [8]:
related_products_df

""


In [9]:
# Basic shape and null checks
summary_df = pd.DataFrame([
    {
        'table': 'agents',
        'rows': len(agents_df),
        'cols': agents_df.shape[1],
        'null_values': int(agents_df.isna().sum().sum()),
    },
    {
        'table': 'dynamic_agent_data',
        'rows': len(dynamic_agent_data_df),
        'cols': dynamic_agent_data_df.shape[1],
        'null_values': int(dynamic_agent_data_df.isna().sum().sum()),
    },
    {
        'table': 'dynamic_product_data',
        'rows': len(dynamic_product_data_df),
        'cols': dynamic_product_data_df.shape[1],
        'null_values': int(dynamic_product_data_df.isna().sum().sum()),
    },
    {
        'table': 'products',
        'rows': len(products_df),
        'cols': products_df.shape[1],
        'null_values': int(products_df.isna().sum().sum()),
    },
    {
        'table': 'related_products',
        'rows': len(related_products_df),
        'cols': related_products_df.shape[1],
        'null_values': int(related_products_df.isna().sum().sum()),
    },
])

summary_df

,table,rows,cols,null_values
0,agents,3,5,0
1,dynamic_agent_data,153,2,0
2,dynamic_product_data,15912,16,0
3,products,104,6,0
4,related_products,0,0,0


In [10]:
# Data consistency checks
checks = {
    'negative_inventory_rows': int((dynamic_product_data_df['inventory'] < 0).sum()),
    'negative_sales_rows': int((dynamic_product_data_df['sales'] < 0).sum()),
    'negative_price_rows': int((dynamic_product_data_df['price'] < 0).sum()),
    'sales_gt_inventory_rows': int((dynamic_product_data_df['sales'] > dynamic_product_data_df['inventory']).sum()),
    'duplicate_agent_step_rows': int(dynamic_agent_data_df.duplicated(['agent_id', 'simulation_step']).sum()),
    'duplicate_agent_product_step_rows': int(dynamic_product_data_df.duplicated(['agent_id', 'product_id', 'simulation_step']).sum()),
}

pd.Series(checks, name='count')

negative_inventory_rows                 0
negative_sales_rows                     0
negative_price_rows                     0
sales_gt_inventory_rows              1249
duplicate_agent_step_rows               0
duplicate_agent_product_step_rows       0
Name: count, dtype: int64

In [11]:
# Agent-level trajectory: cumulative profit over time
# (new exporter no longer includes a direct `balance` field)
agent_step_profit = (
    dynamic_product_data_df
    .groupby(['agent_id', 'simulation_step'], as_index=False)['profit']
    .sum()
    .sort_values(['agent_id', 'simulation_step'])
)
agent_step_profit['cumulative_profit'] = agent_step_profit.groupby('agent_id')['profit'].cumsum()

plt.figure(figsize=(12, 5))
for agent_id, group in agent_step_profit.groupby('agent_id'):
    plt.plot(group['simulation_step'], group['cumulative_profit'], alpha=0.6, label=agent_id)

plt.title('Agent Cumulative Profit Over Time')
plt.xlabel('Simulation Step')
plt.ylabel('Cumulative Profit')
plt.tight_layout()
plt.show()

KeyError: 'balance'

<Figure size 1200x500 with 0 Axes>

In [ ]:
# Aggregate market dynamics: inventory, sales, and orders by step
step_agg = (
    dynamic_product_data_df
    .groupby('simulation_step', as_index=False)[['inventory', 'sales', 'order_quantity', 'outstanding_orders', 'revenue', 'profit']]
    .sum()
)

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(step_agg['simulation_step'], step_agg['inventory'], label='Inventory', color='tab:blue')
axes[0].plot(step_agg['simulation_step'], step_agg['sales'], label='Sales', color='tab:green')
axes[0].plot(step_agg['simulation_step'], step_agg['order_quantity'], label='Order Quantity', color='tab:orange')
axes[0].set_title('Aggregate Units Over Time')
axes[0].set_ylabel('Units')
axes[0].legend()

axes[1].plot(step_agg['simulation_step'], step_agg['revenue'], label='Revenue', color='tab:purple')
axes[1].plot(step_agg['simulation_step'], step_agg['profit'], label='Profit', color='tab:red')
axes[1].set_title('Aggregate Financials Over Time')
axes[1].set_xlabel('Simulation Step')
axes[1].set_ylabel('Amount')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Product-level analysis: top products by cumulative profit
product_perf = (
    dynamic_product_data_df
    .groupby('product_id', as_index=False)[['revenue', 'profit', 'sales']]
    .sum()
    .merge(products_df[['product_id', 'name', 'category']], on='product_id', how='left')
    .sort_values('profit', ascending=False)
)

top_n = 12
top_products = product_perf.head(top_n)

plt.figure(figsize=(12, 6))
sns.barplot(data=top_products, x='profit', y='name', hue='category', dodge=False)
plt.title(f'Top {top_n} Products by Cumulative Profit')
plt.xlabel('Cumulative Profit')
plt.ylabel('Product Name')
plt.legend(title='Category', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

top_products[['product_id', 'name', 'category', 'sales', 'revenue', 'profit']].head(10)

In [ ]:
# Relationship diagnostics: correlation heatmap for core numeric metrics
corr_cols = ['inventory', 'sales', 'order_quantity', 'outstanding_orders', 'price', 'revenue', 'holding_cost', 'total_cost', 'profit']
corr_matrix = dynamic_product_data_df[corr_cols].corr(numeric_only=True)

plt.figure(figsize=(10, 7))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Matrix of Core Product Metrics')
plt.tight_layout()
plt.show()

In [ ]:
# Compact textual summary for quick validation
final_step = dynamic_product_data_df['simulation_step'].max()
final_snapshot = dynamic_product_data_df[dynamic_product_data_df['simulation_step'] == final_step]

summary = {
    'simulation_folder': str(latest_sim_dir),
    'n_agents': int(dynamic_agent_data_df['agent_id'].nunique()),
    'n_products': int(dynamic_product_data_df['product_id'].nunique()),
    'n_steps': int(dynamic_product_data_df['simulation_step'].nunique()),
    'total_revenue': float(dynamic_product_data_df['revenue'].sum()),
    'total_profit': float(dynamic_product_data_df['profit'].sum()),
    'final_total_inventory': int(final_snapshot['inventory'].sum()),
    'final_outstanding_orders': int(final_snapshot['outstanding_orders'].sum()),
}

pd.Series(summary, name='simulation_summary')